# 歌词采集-QQ音乐

In [1]:
import requests
import re
import json
import os
import time
import pandas as pd
import html
from datetime import datetime
from collections import defaultdict


from collections import Counter

In [2]:
import sys
sys.path.append('..')

# 通用方法

## 时间戳格式化

In [3]:
def format_timestamp(ts, date_format='%Y-%m-%d'):
    """
    自动识别秒或毫秒，并转换为指定格式的字符串
    :param ts: 时间戳 (int 或 float)
    """
    if not ts or ts <= 0:
        return "Unknown"
    
    # 核心逻辑：判断时间戳位数
    # 秒级时间戳目前在 10^9 数量级（10位）
    # 毫秒级时间戳在 10^12 数量级（13位）
    # 我们以 10^11 (11位) 为界限进行区分
    if ts > 100000000000: 
        ts = ts / 1000  # 是毫秒，转换为秒
    
    try:
        dt = datetime.fromtimestamp(ts)
        return dt.strftime(date_format)
    except Exception:
        return "Invalid Date"

## 增量保存到json文件

In [4]:
def save_to_json_list(file_path, song_data):
    """以列表形式保存所有歌曲，避免字典 key 覆盖的问题"""
    data_list = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data_list = json.load(f)
                if not isinstance(data_list, list): data_list = []
            except:
                data_list = []

    data_list.append(song_data)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

# 按歌手采集曲目

In [5]:

def search_song(keyword, page=0):
    """搜索歌曲并返回歌曲ID"""
    url = "https://c.y.qq.com/soso/fcgi-bin/client_search_cp"
    params = {
        "w": keyword,
        "format": "json",
        "n": 50,
        "p": page,  
    }
    headers = {
        "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        data = json.loads(response.text)
        songs = data["data"]["song"]["list"]
        res = []
        for song in songs:
            res_d = {
                "song_id": song["songid"],
                "song_mid": song["songmid"],
                "song_name": song["songname"],
                "song_subname": song["lyric"],
                "artist_name": song["singer"][0]["name"],
                "artist_id": song["singer"][0]["id"],
                "artist_mid": song["singer"][0]["mid"],
                "album_name": song['albumname'],
                "album_id": song['albumid'],
                "album_mid": song['albummid'],
                "duration": song['interval'],
                "publish_time": song["pubtime"],
            }
            res.append(res_d)
        return res
    return []

In [6]:
def get_songs_data_raw(singger, max_page=1):
    """
    获取歌手歌曲列表
    singer_name: 歌手名称
    max_page: 最大页数, 默认每页50条数据，max_page=曲目总数/50
    """
    qq_songs_list = []
    for page in range(0, max_page):
        print(f'正在获取第{page+1}页数据...')
        res = search_song(singger, page)
        time.sleep(2)
        qq_songs_list.extend(res)
    return qq_songs_list

# 曲目过滤

## OST曲目筛选

In [7]:
# 数据筛选
# 1. artist_name中包含singger
# 2. song_subname中包含书名号，将书名号中的内容保存为新字段: tv_name
def filter_ost_songs(singger, song_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 子标题包含书名号，并提取书名号内容为 tv_name
    """
    filtered_list = []
    # 预编译正则，匹配《 和 》之间最少的内容
    tv_pattern = re.compile(r'《(.*?)》')

    for song in song_list:
        # 条件 1: 校验 artist_name (确保该字段已在之前的解析中生成)
        if singger not in song.get("artist_name", ""):
            continue

        # 条件 2: 校验 song_name，如果”《“在 song_name 中，则跳过
        if "《" in song.get("song_name", ""):
            continue
            
        # 条件 3: 校验 song_subname 并在满足时提取 tv_name
        subname = song.get("song_subname", "")
        match = tv_pattern.search(subname)
        
        if match:
            # 满足条件，创建新字段并保存
            song["tv_name"] = match.group(1)
            filtered_list.append(song)
            
    return filtered_list

## 按专辑列表

In [18]:
def filter_album_songs(singger, song_list, album_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 专辑包含在 album_list 中
    """
    filtered_list = []

    for song in song_list:
        singers = song.get("artist_name", "")
        album_name = song.get("album_name", "")
        if singger in singers and album_name in album_list:
            filtered_list.append(song)
            
    return filtered_list

## 曲目数据清洗

In [9]:
def clear_song_data(song_data,
                    is_filter_ost=False,
                    is_use_raw_song_name=False):
    """
    清洗歌曲数据
    is_filter_ost: 是否过滤掉OST歌曲
    is_use_raw_song_name: 二次清洗时，是否使用原始歌曲名，例如五月天这种，原始歌曲就有多个版本的，需要在歌名中保留括号内容
    """
    songs_df = pd.DataFrame(song_data)
    if is_use_raw_song_name:
        songs_df['song_name_unique'] = songs_df['song_name']
    else:
        # 分割song_name中的括号
        songs_df['song_name_unique'] = songs_df['song_name'].apply(
            lambda x: x.split('(')[0])
        songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
            lambda x: x.split('（')[0])
        # 删除前后空格
        songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
            lambda x: x.strip())
    # 按song_name_unique进行去重
    songs_df = songs_df.drop_duplicates(subset=['song_name_unique'],
                                        keep='first')
    # 发行时间格式化
    songs_df['publish_date'] = songs_df['publish_time'].apply(
        lambda x: format_timestamp(x))
    if is_filter_ost:
        # 二次筛选ost， 新建列 is_ost, 如果song_subname中含 影，剧，曲任意一个字，则is_ost为1，否则为0
        songs_df['is_ost'] = songs_df['song_subname'].apply(
            lambda x: 1 if '影' in x or '剧' in x or '曲' in x or '片' in x else 0)
        songs_df = songs_df[songs_df['is_ost'] == 1]
    return songs_df

## 专辑信息清洗

In [10]:
# 专辑数据清洗
def clear_album_data(songs_data):
    album_count = songs_data.groupby(
        ['album_name',
         'album_id'])['album_id'].count().reset_index(name='count')
    album_count = album_count.sort_values(by=['count'], ascending=False)
    # 取count最大值所在行的数据为album_id
    album_id = album_count.drop_duplicates(subset=['album_name'],
                                           keep='first').reset_index(drop=True)
    # 匹配专辑发行日期
    album_date = songs_data[['album_id', 'publish_date']].drop_duplicates(
        subset=['album_id'], keep='first').reset_index(drop=True)
    album_df = album_id[['album_name', 'album_id']].merge(album_date,
                                                          on='album_id',
                                                          how='left')
    songs_data_cleared = songs_data.drop(['album_id', 'publish_date'],
                                         axis=1).copy()
    songs_data_cleared = songs_data_cleared.merge(album_df,
                                                  on='album_name',
                                                  how='left')
    return songs_data_cleared

# 歌词采集

In [11]:
def get_qq_lyric(song_id):
    """根据歌曲ID获取歌词"""
    url = "https://c.y.qq.com/lyric/fcgi-bin/fcg_query_lyric_yqq.fcg"
    params = {
        "nobase64": 1,
        "musicid": song_id,
        "format": "json"
    }
    headers = {
        "Referer": "https://y.qq.com/",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        lyric_data = response.json()
        return lyric_data.get("lyric", "")
    return "歌词获取失败"

In [12]:
def get_all_songs_lyric(file_path, songs_df):
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        # 判断是否已采集
        if os.path.exists(file_path):
            df = pd.read_json(file_path)
            songs_had = df['song_id'].tolist()
            if song_id not in songs_had:
                print(song_name)
                lyric_raw = get_qq_lyric(song_id)
                single_res = {
                    'song_id': song_id,
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
                save_to_json_list(file_path, single_res)
                time.sleep(2)
        else:
            print(song_name)
            lyric_raw = get_qq_lyric(song_id)
            single_res = {
                    'song_id': song_id,
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
            save_to_json_list(file_path, single_res)
            time.sleep(2)

## 歌词清洗

In [13]:
class QQLyricCleaner:
    def __init__(self):
        self.time_tag_pattern = re.compile(r'\[\d{2,}:\d{2,}\.\d{2,}\]\s*(.*)')
        self.isrc_pattern = re.compile(r'[A-Z]{2}-[A-Z0-9]{3}-\d{2}-\d{5}')
        
        # 扩展黑名单：不仅包含职位，还包含法律/版权声明
        self.exclude_keywords = [
            '词', 'Lyricist', '曲', 'Composer', '编', 'Arranger', 
            '制作', 'Producer', '执行', 'Executive', '合音', 'Chorus',
            '鼓', 'Drums', '钢琴', 'Piano', '录音', 'Engineer', 'Studio',
            '混音', 'Mixing', '母带', 'Mastering', 'OP', 'SP', '版权',
            '：', ':', 'ISRC', '提供', '发行', '编码', 'BY:',
            '统筹', '营销', '推广', '监制', '出品', '弦乐', '唢呐', '和声', '艺人',
            '著作权', '未经', '许可', '翻唱', '翻录', '使用', '不得', '权利人'
        ]

    def _preprocess(self, raw_text):
        if not raw_text: return ""
        # 处理 HTML 实体字符
        text = html.unescape(raw_text)
        # 统一换行与空格
        text = text.replace('&#10;', '\n').replace('\r', '')
        text = text.replace('\u00a0', ' ').replace('&#32;', ' ')
        return text

    def get_credits(self, text):
        """提取制作人信息：作词、作曲、编曲"""
        text = self._preprocess(text)
        credits = {"lyricist": "", "composer": "", "arranger": ""}
        mapping = {
            "lyricist": r"(?:词|作词)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)",
            "composer": r"(?:曲|作曲)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)",
            "arranger": r"(?:编曲|Arranger)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)"
        }
        for key, pattern in mapping.items():
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                # 剔除后面可能跟着的时间戳或括号
                val = re.split(r'[\[\]]', match.group(1).strip())[0].strip()
                credits[key] = val
        return credits

    def get_pure_lyrics(self, raw_text, song_name=None):
        """清洗歌词主体：剔除标题、幕后信息、广告、版权声明"""
        text = self._preprocess(raw_text)
        if not text: return ""

        lines = text.split('\n')
        pure_lyrics = []

        for line in lines:
            match = self.time_tag_pattern.search(line)
            if match:
                content = match.group(1).strip()
                if not content: continue
                
                # --- 核心过滤逻辑 ---
                
                # 1. 过滤特殊括号和推广标签 (如星曜计划)
                if any(tag in content for tag in ['『', '』', '星曜计划']):
                    continue
                
                # 2. 标题行过滤 (命中歌名或包含 歌手+书名号)
                is_title = False
                if song_name and song_name in content: is_title = True
                if '《' in content and '》' in content and ('刘宇宁' in content or '摩登兄弟' in content): is_title = True
                if '-' in content and ('刘宇宁' in content or '摩登兄弟' in content): is_title = True
                if is_title and len(content) < 45: continue

                # 3. 幕后职位及版权声明过滤 (检测行首 15 字)
                content_prefix = content[:15]
                if any(k in content_prefix for k in self.exclude_keywords):
                    continue
                
                # 4. 特殊清洗：如果行内残余“未经著作权人...”这种特定文本，直接跳过
                if re.search(r'未经著作权人|不得翻唱|翻录或使用', content):
                    continue

                if self.isrc_pattern.search(content): continue

                # 5. 格式化：内部空格转逗号
                clean_line = re.sub(r'\s+', ' ', content).replace(" ", "，")
                pure_lyrics.append(clean_line)

        # 返回以句号连接的歌词
        return "。".join(pure_lyrics) + "。" if pure_lyrics else ""

    def process_data(self, raw_lyric, song_id=None, song_name=None):
        """主入口"""
        credits = self.get_credits(raw_lyric)
        lyrics_text = self.get_pure_lyrics(raw_lyric, song_name=song_name)
        
        return {
            "song_id": song_id,
            "song_name": song_name,
            "has_lyric": 1 if lyrics_text else 0,
            "lyricist": credits["lyricist"],
            "composer": credits["composer"],
            "arranger": credits["arranger"],
            "lyrics_text": lyrics_text
        }

In [92]:
# 优化版
class QQLyricCleaner:
    def __init__(self):
        # 匹配时间戳和内容
        self.time_tag_pattern = re.compile(r'\[\d{2,}:\d{2,}\.\d{2,}\]\s*(.*)')
        self.isrc_pattern = re.compile(r'[A-Z]{2}-[A-Z0-9]{3}-\d{2}-\d{5}')
        
        # 扩展黑名单
        self.exclude_keywords = [
            '词', 'Lyricist', '曲', 'Composer', '编', 'Arranger', 
            '制作', 'Producer', '执行', 'Executive', '合音', 'Chorus',
            '鼓', 'Drums', '钢琴', 'Piano', '录音', 'Engineer', 'Studio',
            '混音', 'Mixing', '母带', 'Mastering', 'OP', 'SP', '版权',
            'ISRC', '提供', '发行', '编码', 'BY:',
            '统筹', '营销', '推广', '监制', '出品', '弦乐', '唢呐', '和声', '艺人',
            '著作权', '未经', '许可', '翻唱', '翻录', '使用', '不得', '权利人'
        ]

    def _preprocess(self, raw_text):
        if not raw_text: return ""
        # 1. 先进行 HTML 反转义 (如 &#58; -> :)
        text = html.unescape(raw_text)
        # 2. 统一处理换行与各种特殊空格
        text = text.replace('\r', '')
        text = re.sub(r'&#10;|\n', '\n', text)
        text = re.sub(r'&#32;|\u00a0', ' ', text)
        return text

    def get_credits(self, text):
        text = self._preprocess(text)
        credits = {"lyricist": "", "composer": "", "arranger": ""}
        mapping = {
            "lyricist": r"(?:词|作词)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)",
            "composer": r"(?:曲|作曲)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)",
            "arranger": r"(?:编曲|Arranger)(?:\s*[a-zA-Z]+)?\s*[:：]\s*([^\n\r\]]+)"
        }
        for key, pattern in mapping.items():
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                val = re.split(r'[\[\]]', match.group(1).strip())[0].strip()
                credits[key] = val
        return credits

    def get_pure_lyrics(self, raw_text, song_name=None):
        text = self._preprocess(raw_text)
        if not text: return ""

        lines = text.split('\n')
        pure_lyrics = []

        for line in lines:
            match = self.time_tag_pattern.search(line)
            if match:
                content = match.group(1).strip()
                if not content: continue
                
                # --- 优化后的过滤逻辑 ---
                
                # 1. 过滤特殊推广标签
                if any(tag in content for tag in ['『', '』', '星曜计划']):
                    continue
                
                # 2. 修正后的标题行过滤
                # 只有当行内容完全等于歌名，或者呈现 "歌名 - 歌手" 的标题格式时才过滤
                if song_name:
                    content_clean = content.replace(" ", "")
                    sn_clean = song_name.replace(" ", "")
                    # 如果内容完全匹配歌名，跳过
                    if content_clean == sn_clean: continue
                    # 如果是 "歌名 - 歌手" 或 "歌手 - 歌名" 这种标题行常用的连字符格式，跳过
                    if " - " in content and sn_clean in content_clean: continue

                # 3. 幕后职位及版权声明过滤 (更加严格的开头匹配)
                # 使用冒号或特定关键词判断是否为信息行
                if any(content.startswith(k) for k in self.exclude_keywords) or \
                   any(f"{k}:" in content.upper() for k in ['词', '曲', 'OP', 'SP']):
                    continue
                
                # 4. 法律条款扫描
                if re.search(r'未经著作权人|不得翻唱|翻录或使用|本歌词由', content):
                    continue

                if self.isrc_pattern.search(content): continue

                # 5. 格式化：内部空格处理
                # 如果你想保持歌词中间的停顿，建议转为逗号；如果不需要，直接合拢
                clean_line = re.sub(r'\s+', ' ', content).strip()
                # 只有当歌词中间确实有空格时，才替换为逗号（如：突然好想你 你会在哪里）
                clean_line = clean_line.replace(" ", "，")
                
                pure_lyrics.append(clean_line)

        # 返回以句号连接的歌词
        return "。".join(pure_lyrics) + "。" if pure_lyrics else ""

    def process_data(self, raw_lyric, song_id=None, song_name=None):
        credits = self.get_credits(raw_lyric)
        lyrics_text = self.get_pure_lyrics(raw_lyric, song_name=song_name)
        
        return {
            "song_id": song_id,
            "song_name": song_name,
            "has_lyric": 1 if lyrics_text else 0,
            "lyricist": credits["lyricist"],
            "composer": credits["composer"],
            "arranger": credits["arranger"],
            "lyrics_text": lyrics_text
        }

In [93]:
def clear_and_save_lyric(path_prefix, songs_df):
    lyric_raw = pd.read_json(path_prefix+'raw_lyric_data.json')

    cleaner = QQLyricCleaner()
    res_list = []
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        lyric_raw_single = lyric_raw[lyric_raw['song_id'] == song_id]['lyric_raw'].values[0]
        single_res = cleaner.process_data(lyric_raw_single, song_id, song_name)
        res_list.append(single_res)
    with open(path_prefix+'cleared_lyric_data.json', 'w', encoding='utf-8') as f:
        json.dump(res_list, f, ensure_ascii=False, indent=4)

# main

## 歌手

In [15]:
# file_path_prefix = "data/jaychou/"
# singger = "周杰伦"
# max_page = 20

file_path_prefix = "data/mayday/"
singger = "五月天"
max_page = 16

### 曲目采集

In [25]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singger, max_page=max_page)

正在获取第1页数据...
正在获取第2页数据...
正在获取第3页数据...
正在获取第4页数据...
正在获取第5页数据...
正在获取第6页数据...
正在获取第7页数据...
正在获取第8页数据...
正在获取第9页数据...
正在获取第10页数据...
正在获取第11页数据...
正在获取第12页数据...
正在获取第13页数据...
正在获取第14页数据...
正在获取第15页数据...
正在获取第16页数据...


In [27]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [69]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [70]:
# 曲目筛选，按专辑
# 周杰伦
album_list = [
    "Jay", "范特西", "八度空间", "叶惠美", "七里香", "十一月的肖邦", "依然范特西", "我很忙", "魔杰座", "跨时代",
    "惊叹号", "十二新作", "哎哟，不错哦", "周杰伦的床边故事", "最伟大的作品"
]
# 五月天
album_list = [
    '第一张创作专辑', '爱情万岁', '人生海海', '时光机', '神的孩子都在跳舞', '为爱而生', '后青春期的诗',
    '第二人生（明日版）', '第二人生（末日版）', '自传', '知足 最真杰作选', '步步 自选作品辑 the Best of 1999-2013'
]
song_data_filted = filter_album_songs(singger, song_data_raw_read, album_list)

In [71]:
len(song_data_filted)

174

### 数据验证
检查专辑中歌曲是否存在缺失，将缺失歌曲的数据手工添加到song_data_filted中

In [72]:
song_data_cleared_1 = clear_song_data(song_data_filted)
song_data_cleared_1

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,1393445,002fRO0N4FftzY,346,1469030400,后来的我们,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,36459,0020I7sO0ayXhN,265,1224691200,突然好想你,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑 the Best of 1999-2013,451706,0006MmDz4Hl2Ud,273,1388332800,步步,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,96397,003PIMo40rxcAn,256,1124985600,知足,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,1393445,002fRO0N4FftzY,249,1469030400,派对动物,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,4931459,002isxS21aW7nO,啾啾啾,NaN,五月天,74,000Sp0Bz4JXH0o,人生海海,96291,0032BHCb4D2LQX,229,994435200,啾啾啾,2001-07-07
167,182034,000NIye52bUSrt,前传,NaN,五月天,74,000Sp0Bz4JXH0o,为爱而生,15702,000hWSug0b5y3S,44,1167321600,前传,2006-12-29
169,1056386,001wlNGy3vEMkl,明日,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,37,1323964800,明日,2011-12-16
171,4932782,004R6eJB0pQmfu,"未来Sailing, With Me",NaN,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,96397,003PIMo40rxcAn,79,1124985600,"未来Sailing, With Me",2005-08-26


In [73]:
song_data_cleared_1.groupby('album_name')['song_name'].count()

album_name
为爱而生                              13
人生海海                              12
后青春期的诗                            12
时光机                               14
步步 自选作品辑 the Best of 1999-2013     5
爱情万岁                              12
知足 最真杰作选                          10
神的孩子都在跳舞                          12
第一张创作专辑                           12
第二人生（明日版）                          7
第二人生（末日版）                          7
自传                                13
Name: song_name, dtype: int64

In [48]:
# 问题专辑
# 周杰伦
album_list_to_fix = ['最伟大的作品']
# 五月天
album_list_to_fix = ['第二人生（明日版）']
song_data_cleared_1[song_data_cleared_1['album_name'] == album_list_to_fix[0]]

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
5,1056382,003Pi1MC2SskqJ,干杯,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,289,1323964800,干杯,2011-12-16
11,1056383,001A1Rev26Yzah,仓颉,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,301,1323964800,仓颉,2011-12-16
14,1056519,000xFQ7C3VJCb3,OAOA (现在就是永远),《五月天追梦3DNA》电影主题曲,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,282,1323964800,OAOA,2011-12-16
83,1056520,003R3hpD2iXWUg,诺亚方舟,《五月天2012诺亚方舟世界巡回演唱会》主题曲,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,339,1323964800,诺亚方舟,2011-12-16
117,1056380,003T9WSk1dV6z2,三个傻瓜,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,256,1323964800,三个傻瓜,2011-12-16
122,1056381,004W17Er3PGk8E,歪腰,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,189,1323964800,歪腰,2011-12-16
169,1056386,001wlNGy3vEMkl,明日,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生（明日版）,88493,001fbipy4azgKM,37,1323964800,明日,2011-12-16


In [75]:
# 补充歌曲，并修改专辑信息
# 周杰伦
songs_list_to_add = ['说好不哭', '不爱我就拉倒', 'Mojito', '等你下课', '我是如此相信', '英雄']
# 五月天
songs_dict_to_add = {'生命有一种绝对': '时光机', 'Enrich Your Life': '神的孩子都在跳舞', '垃圾车 (朋友版)': '神的孩子都在跳舞', '温柔 (还你自由版)': '知足 最真杰作选', '入阵曲': '步步 自选作品辑 the Best of 1999-2013', '离开地球表面': '步步 自选作品辑 the Best of 1999-2013', 'OAOA (丢掉名字性别)': '第二人生（末日版）', '我不愿让你一个人': '第二人生（末日版）'}
for i in song_data_raw:
    if i['song_name'] in songs_dict_to_add.keys():
        print(i)

{'song_id': 405385, 'song_mid': '001BAFqt1Ay4Vf', 'song_name': '离开地球表面', 'song_subname': '《开心超人》动画电影片尾曲', 'artist_name': '五月天', 'artist_id': 74, 'artist_mid': '000Sp0Bz4JXH0o', 'album_name': '离开地球表面 Jump!', 'album_id': 32775, 'album_mid': '002PYDbl3I5L2k', 'duration': 275, 'publish_time': 1184860800}
{'song_id': 4996096, 'song_mid': '003Xy9E32vvMLe', 'song_name': '入阵曲', 'song_subname': '《兰陵王》电视剧主题曲', 'artist_name': '五月天', 'artist_id': 74, 'artist_mid': '000Sp0Bz4JXH0o', 'album_name': '兰陵王 电视剧原声带', 'album_id': 431765, 'album_mid': '002adz882rV5uh', 'duration': 209, 'publish_time': 1377792000}
{'song_id': 405385, 'song_mid': '001BAFqt1Ay4Vf', 'song_name': '离开地球表面', 'song_subname': '《开心超人》动画电影片尾曲', 'artist_name': '五月天', 'artist_id': 74, 'artist_mid': '000Sp0Bz4JXH0o', 'album_name': '离开地球表面 Jump!', 'album_id': 32775, 'album_mid': '002PYDbl3I5L2k', 'duration': 275, 'publish_time': 1184860800}
{'song_id': 4996096, 'song_mid': '003Xy9E32vvMLe', 'song_name': '入阵曲', 'song_subname': '《兰陵王》电视剧主题曲

In [ ]:
# 手工添加需要补全的歌曲
# 周杰伦
songs_to_add = [{
    'song_id': 105755384,
    'song_mid': '004Qscj80GYhGR',
    'song_name': '英雄',
    'song_subname': '《英雄联盟》中国品牌主题曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '周杰伦的床边故事',
    'album_id': 1306793,
    'album_mid': '001uJFiE0tbGGa',
    'duration': 200,
    'publish_time': 1458748800
}, {
    'song_id': 247261229,
    'song_mid': '001PLl3C4gPSCI',
    'song_name': '我是如此相信',
    'song_subname': '《天火》电影主题曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 9612009,
    'album_mid': '001hGx1Z0so1YX',
    'duration': 266,
    'publish_time': 1576339200
}, {
    'song_id': 268352018,
    'song_mid': '001glaI72k8BQX',
    'song_name': 'Mojito',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 12924001,
    'album_mid': '0009C3rp3Kfwg0',
    'duration': 185,
    'publish_time': 1591891200
}, {
    'song_id': 213922043,
    'song_mid': '0031TAKo0095np',
    'song_name': '不爱我就拉倒',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 4044657,
    'album_mid': '001CnPE31iJ899',
    'duration': 245,
    'publish_time': 1526313600
}, {
    'song_id': 212877900,
    'song_mid': '001J5QJL1pRQYB',
    'song_name': '等你下课 (with 杨瑞代)',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 3883404,
    'album_mid': '003bSL0v4bpKAx',
    'duration': 270,
    'publish_time': 1516204800
}, {
    'song_id': 237773700,
    'song_mid': '001qvvgF38HVc4',
    'song_name': '说好不哭 (with 五月天阿信)',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 7876962,
    'album_mid': '002gBTVk4JEE2T',
    'duration': 222,
    'publish_time': 1568646000
}]

In [76]:
# 五月天
songs_to_add = [
    {
        'song_id': 405385,
        'song_mid': '001BAFqt1Ay4Vf',
        'song_name': '离开地球表面',
        'song_subname': '《开心超人》动画电影片尾曲',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '离开地球表面 Jump!',
        'album_id': 32775,
        'album_mid': '002PYDbl3I5L2k',
        'duration': 275,
        'publish_time': 1184860800
    },
    {
        'song_id': 4996096,
        'song_mid': '003Xy9E32vvMLe',
        'song_name': '入阵曲',
        'song_subname': '《兰陵王》电视剧主题曲',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '兰陵王 电视剧原声带',
        'album_id': 431765,
        'album_mid': '002adz882rV5uh',
        'duration': 209,
        'publish_time': 1377792000
    },
    {
        'song_id': 4932058,
        'song_mid': '003PaRAX3j5wJk',
        'song_name': '生命有一种绝对',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '摇滚本事 电影音乐原声带',
        'album_id': 96335,
        'album_mid': '0015r2I31enfaR',
        'duration': 239,
        'publish_time': 1038672000
    },
    {
        'song_id': 4830242,
        'song_mid': '000PoJAV4NPMzW',
        'song_name': '温柔 (还你自由版)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '音乐电影-五月之恋',
        'album_id': 96361,
        'album_mid': '001ntd0y01uQ4g',
        'duration': 426,
        'publish_time': 1088611200
    },
    {
        'song_id': 1056504,
        'song_mid': '002Sv0dp3T3p5U',
        'song_name': 'OAOA (丢掉名字性别)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '第二人生（末日版）',
        'album_id': 90142,
        'album_mid': '000IPRft1LSCqL',
        'duration': 282,
        'publish_time': 1323964800
    },
    {
        'song_id': 4834459,
        'song_mid': '002nqyCb1bUnk6',
        'song_name': 'Enrich Your Life',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': 'Enrich Your Life',
        'album_id': 62890,
        'album_mid': '0006oAnx03zXUC',
        'duration': 166,
        'publish_time': 1096560000
    },
    {
        'song_id': 4932456,
        'song_mid': '002qi7L00Cpb73',
        'song_name': '垃圾车 (朋友版)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '神的孩子都在跳舞',
        'album_id': 96368,
        'album_mid': '002plCgA0zOyYF',
        'duration': 243,
        'publish_time': 1099238400
    },
    {'song_id': 519403016, 'song_mid': '000B69Qg0S8WUF', 'song_name': '我不愿让你一个人', 'song_subname': '《今夜一起为爱鼓掌》电视剧插曲', 'artist_name': '五月天', 'artist_id': 74, 'artist_mid': '000Sp0Bz4JXH0o', 'album_name': '今夜一起为爱鼓掌 电视剧原声带', 'album_id': 56247634, 'album_mid': '001Jhk1t0SC1FZ', 'duration': 265, 'publish_time': 1727625600}

]
# 修改专辑名称
songs_to_add_fixed = []
for song in songs_to_add:
    song['album_name'] = songs_dict_to_add[song['song_name']]
    songs_to_add_fixed.append(song)

### 专辑名称手工修改

In [77]:
# 五月天
album_to_fix_dict = {
    '第二人生（末日版）': '第二人生',
    '第二人生（明日版）': '第二人生',
    '步步 自选作品辑 the Best of 1999-2013': '步步 自选作品辑',
}

In [78]:
song_data_filted.extend(songs_to_add_fixed)
for i in song_data_filted:
    if i['album_name'] in album_to_fix_dict:
        i['album_name'] = album_to_fix_dict[i['album_name']]

In [79]:
len(song_data_filted)

182

### 二次清洗

In [80]:
# song_data_filted.extend(songs_to_add)
# 五月天歌曲名使用原始歌曲名
song_data_cleared_2 = clear_song_data(song_data_filted, is_use_raw_song_name=True)
song_data_cleared_2

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,1393445,002fRO0N4FftzY,346,1469030400,后来的我们,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,36459,0020I7sO0ayXhN,265,1224691200,突然好想你,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,451706,0006MmDz4Hl2Ud,273,1388332800,步步,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,96397,003PIMo40rxcAn,256,1124985600,知足,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,1393445,002fRO0N4FftzY,249,1469030400,派对动物,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,431765,002adz882rV5uh,209,1377792000,入阵曲,2013-08-30
176,4932058,003PaRAX3j5wJk,生命有一种绝对,,五月天,74,000Sp0Bz4JXH0o,时光机,96335,0015r2I31enfaR,239,1038672000,生命有一种绝对,2002-12-01
177,4830242,000PoJAV4NPMzW,温柔 (还你自由版),,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,96361,001ntd0y01uQ4g,426,1088611200,温柔 (还你自由版),2004-07-01
179,4834459,002nqyCb1bUnk6,Enrich Your Life,,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,62890,0006oAnx03zXUC,166,1096560000,Enrich Your Life,2004-10-01


In [81]:
song_data_cleared_2.groupby('album_name')['song_name'].count()

album_name
为爱而生        13
人生海海        12
后青春期的诗      12
时光机         15
步步 自选作品辑    11
爱情万岁        12
知足 最真杰作选    11
神的孩子都在跳舞    14
第一张创作专辑     12
第二人生        17
自传          13
Name: song_name, dtype: int64

In [83]:
song_data_cleared_2[song_data_cleared_2['album_name'] == '第二人生']

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
5,1056382,003Pi1MC2SskqJ,干杯,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生,88493,001fbipy4azgKM,289,1323964800,干杯,2011-12-16
11,1056383,001A1Rev26Yzah,仓颉,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生,88493,001fbipy4azgKM,301,1323964800,仓颉,2011-12-16
14,1056519,000xFQ7C3VJCb3,OAOA (现在就是永远),《五月天追梦3DNA》电影主题曲,五月天,74,000Sp0Bz4JXH0o,第二人生,88493,001fbipy4azgKM,282,1323964800,OAOA (现在就是永远),2011-12-16
19,1056501,002tNi6t3aj8RP,星空,《星空》电影同名主题曲,五月天,74,000Sp0Bz4JXH0o,第二人生,90142,000IPRft1LSCqL,283,1323964800,星空,2011-12-16
33,1056505,002e1Ddt39PqKm,第二人生,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生,90142,000IPRft1LSCqL,269,1323964800,第二人生,2011-12-16
83,1056520,003R3hpD2iXWUg,诺亚方舟,《五月天2012诺亚方舟世界巡回演唱会》主题曲,五月天,74,000Sp0Bz4JXH0o,第二人生,88493,001fbipy4azgKM,339,1323964800,诺亚方舟,2011-12-16
93,1056507,003WzxcD24fESC,有些事现在不做 一辈子都不会做了,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生,90142,000IPRft1LSCqL,228,1323964800,有些事现在不做 一辈子都不会做了,2011-12-16
99,1056497,000qSMBu3mcwM6,洗衣机,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生,90142,000IPRft1LSCqL,243,1323964800,洗衣机,2011-12-16
107,1056503,000cZufl0dGIWL,末日,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生,90142,000IPRft1LSCqL,4,1323964800,末日,2011-12-16
117,1056380,003T9WSk1dV6z2,三个傻瓜,NaN,五月天,74,000Sp0Bz4JXH0o,第二人生,88493,001fbipy4azgKM,256,1323964800,三个傻瓜,2011-12-16


In [84]:
# 需要手工删除的歌
songs_to_drop = [
    'T1 21 31 21 (Bonus Track)', '知足 (乐团版)', '拥抱 (2013新录制作品)',
    '温柔 (2013Remix版)', '憨人 (Live)'
]
# 删除song_data_cleared中song_name在songs_to_drop的行
song_data_cleared = song_data_cleared_2[~song_data_cleared_2['song_name'].
                                         isin(songs_to_drop)]
song_data_cleared

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,1393445,002fRO0N4FftzY,346,1469030400,后来的我们,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,36459,0020I7sO0ayXhN,265,1224691200,突然好想你,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,451706,0006MmDz4Hl2Ud,273,1388332800,步步,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,96397,003PIMo40rxcAn,256,1124985600,知足,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,1393445,002fRO0N4FftzY,249,1469030400,派对动物,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,431765,002adz882rV5uh,209,1377792000,入阵曲,2013-08-30
176,4932058,003PaRAX3j5wJk,生命有一种绝对,,五月天,74,000Sp0Bz4JXH0o,时光机,96335,0015r2I31enfaR,239,1038672000,生命有一种绝对,2002-12-01
177,4830242,000PoJAV4NPMzW,温柔 (还你自由版),,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,96361,001ntd0y01uQ4g,426,1088611200,温柔 (还你自由版),2004-07-01
179,4834459,002nqyCb1bUnk6,Enrich Your Life,,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,62890,0006oAnx03zXUC,166,1096560000,Enrich Your Life,2004-10-01


In [85]:
song_data_cleared_final = clear_album_data(song_data_cleared)
song_data_cleared_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,0020I7sO0ayXhN,265,1224691200,突然好想你,36459,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,0006MmDz4Hl2Ud,273,1388332800,步步,451706,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,003PIMo40rxcAn,256,1124985600,知足,96397,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,249,1469030400,派对动物,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,002adz882rV5uh,209,1377792000,入阵曲,451706,2013-12-30
133,4932058,003PaRAX3j5wJk,生命有一种绝对,,五月天,74,000Sp0Bz4JXH0o,时光机,0015r2I31enfaR,239,1038672000,生命有一种绝对,96353,2003-11-07
134,4830242,000PoJAV4NPMzW,温柔 (还你自由版),,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,001ntd0y01uQ4g,426,1088611200,温柔 (还你自由版),96397,2005-08-26
135,4834459,002nqyCb1bUnk6,Enrich Your Life,,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,0006oAnx03zXUC,166,1096560000,Enrich Your Life,96368,2004-11-01


In [86]:
song_data_cleared_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

### 歌词采集

In [87]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared)

我不愿让你一个人


### 歌词清洗

In [94]:
clear_and_save_lyric(file_path_prefix, song_data_cleared)